# TOBB Price Regression

**ret_usd ~ shortfall_trail** — annual log USD price return regressed on trailing 5yr production shortfall.

Price series: TOBB harvest-season (Aug–Oct) VWAP, all exchanges, volume-weighted, converted to USD.
Gap years 2020–2022 filled from crop-year Giresun file (also TOBB data).

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from tobb_price_regression import build_price_series, build_regression_dataset, run_models

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})

price  = build_price_series()
df     = build_regression_dataset(price)
models = run_models(df)

print(f"Price series: {len(price)} years ({price['year'].min()}–{price['year'].max()})")
print(f"Regression dataset: n={len(df)} ({df['year'].min()}–{df['year'].max()})")
print()
print(price[['year','vwap_try','vwap_usd','n_days','source_flag']].to_string(index=False))

## 1 · Price series — TL/kg and USD/kg

Left: nominal TL price (inflated by TRY depreciation post-2018). Right: USD equivalent — the economically meaningful series.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))

# mark gap-filled years
scraped  = price[price['source_flag'] == 'tobb_scraped']
fallback = price[price['source_flag'] == 'tobb_cropyear']

ax1.plot(price['year'], price['vwap_try'], '-o', ms=4, lw=1.2, color='steelblue', label='scraped')
if not fallback.empty:
    ax1.scatter(fallback['year'], fallback['vwap_try'], color='orange', zorder=5, label='crop-year fill')
ax1.set_title('Harvest VWAP — TL/kg')
ax1.set_ylabel('TL / kg')
ax1.legend(fontsize=8)

ax2.plot(price['year'], price['vwap_usd'], '-o', ms=4, lw=1.2, color='steelblue', label='scraped')
if not fallback.empty:
    ax2.scatter(fallback['year'], fallback['vwap_usd'], color='orange', zorder=5, label='crop-year fill')
ax2.set_title('Harvest VWAP — USD/kg')
ax2.set_ylabel('USD / kg')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2 · Shortfall vs price return

Scatter: x = trailing 5yr shortfall (%), y = annual log USD price return. Each point is one year.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

gap_mask = df['source_flag'] == 'tobb_cropyear'
ax.scatter(df.loc[~gap_mask, 'shortfall_trail'], df.loc[~gap_mask, 'ret_usd'],
           s=40, color='steelblue', label='scraped', zorder=3)
ax.scatter(df.loc[gap_mask, 'shortfall_trail'], df.loc[gap_mask, 'ret_usd'],
           s=40, color='orange', marker='D', label='crop-year fill', zorder=3)

for _, row in df.iterrows():
    ax.annotate(str(int(row['year'])), (row['shortfall_trail'], row['ret_usd']),
                fontsize=7, xytext=(3, 3), textcoords='offset points', color='gray')

# M1 fit line
m1  = models['M1_shortfall']
xs  = np.linspace(df['shortfall_trail'].min(), df['shortfall_trail'].max(), 100)
ys  = m1.params['intercept'] + m1.params['shortfall_trail'] * xs
ax.plot(xs, ys, '--', color='crimson', lw=1.2, label='M1 fit')

ax.axhline(0, color='black', lw=0.5)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Trailing 5yr shortfall (%)')
ax.set_ylabel('Annual log USD price return')
ax.set_title('Price return vs production shortfall')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3 · Model results

In [ ]:
rows = []
for name, m in models.items():
    tbl = pd.DataFrame({
        'coef':  m.params.round(4),
        't':     m.tvalues.round(2),
        'p':     m.pvalues.round(3),
    })
    print(f'\n--- {m.summary_line()} ---')
    print(tbl.to_string())

# Comparison table
comp = pd.DataFrame([
    {'model': m.name, 'n': m.n, 'R²': round(m.rsquared, 3),
     'adj-R²': round(m.rsquared_adj, 3), 'AIC': round(m.aic, 1)}
    for m in models.values()
])
print()
print(comp.to_string(index=False))

## 4 · Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

for ax, (name, m) in zip(axes, models.items()):
    n = m.n
    years_used = df['year'].values[-n:] if name == 'M1_shortfall' else df['year'].values[-(n):]
    ax.bar(range(n), m.resid, color=['crimson' if r < 0 else 'steelblue' for r in m.resid])
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{name}  |  residuals')
    ax.set_xlabel('observation')
    ax.set_ylabel('residual')

plt.tight_layout()
plt.show()

## Notes

- **n=18**: TOBB scrape covers 2005–2025 harvest but 2021–22 are absent from the exchange (real gap). With first-diff for returns, n=18 usable years.
- **R²=0.367 vs 0.529** in the annual panel model: the existing panel used a longer series (1990s–present with FAO production data extending the shortfall series back further). TOBB prices only available from 2005, shortening the window.
- **Shortfall coefficient ≈ −0.010**: a 10pp below-trend harvest → ~10% USD price increase. Consistent in sign and magnitude with the longer panel.
- **M2 (+ price lag)**: lag is not significant (p=0.29). No evidence of serial correlation in price returns beyond the shortfall signal.